# 1. Imports

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    monotonically_increasing_id,
    col
)

# 2. Configurações Globais

In [0]:
# ============================================================
# CONFIGURAÇÕES DO PROJETO
# ============================================================

CATALOG = "workspace"

SCHEMA_BRONZE = "bronze"

VOLUME_PATH = "/Volumes/workspace/default/tech_challenge_alfabetizacao"

FORMATO_ARQUIVO = "csv"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}
""")

DataFrame[]

In [0]:
arquivos_csv = [
    arquivo.name
    for arquivo in dbutils.fs.ls(VOLUME_PATH)
    if arquivo.name.endswith(".csv")
]

# Função Ingestão Bronze

In [0]:
def carregar_arquivo_bronze(nome_arquivo: str) -> DataFrame:

    caminho_arquivo = f"{VOLUME_PATH}/{nome_arquivo}"

    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("sep", ",")
        .option("encoding", "UTF-8")
        .option("inferSchema", False)
        .load(caminho_arquivo)
    )

    df_bronze = (
        df
        .withColumn("_id_linha", monotonically_increasing_id())
        .withColumn("_data_ingestao", current_timestamp())
        .withColumn("_arquivo_origem", lit(nome_arquivo))
        .withColumn("_caminho_arquivo", col("_metadata.file_path"))
        .withColumn("_camada", lit("bronze"))
    )

    return df_bronze

# Função escrita Delta

In [0]:
# ============================================================
# FUNÇÃO DE ESCRITA DELTA
# ============================================================

def salvar_tabela_bronze(df: DataFrame, nome_tabela: str):

    tabela_destino = f"{CATALOG}.{SCHEMA_BRONZE}.{nome_tabela}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    print(f"Tabela salva: {tabela_destino}")

# Pipeline da Ingestão

In [0]:
# ============================================================
# PIPELINE DE INGESTÃO BRONZE
# ============================================================

for arquivo in arquivos_csv:

    nome_tabela = arquivo.replace(".csv", "")

    print("=" * 80)
    print(f"Iniciando ingestão: {arquivo}")

    df_bronze = carregar_arquivo_bronze(arquivo)

    salvar_tabela_bronze(df_bronze, nome_tabela)

    total_registros = df_bronze.count()

    print(f"Total registros: {total_registros}")
    print(f"Ingestão finalizada: {nome_tabela}")

Iniciando ingestão: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv
Tabela salva: workspace.bronze.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil
Total registros: 3
Ingestão finalizada: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil
Iniciando ingestão: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv
Tabela salva: workspace.bronze.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio
Total registros: 10704
Ingestão finalizada: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio
Iniciando ingestão: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv
Tabela salva: workspace.bronze.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf
Total registros: 54
Ingestão finalizada: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf
Iniciando ingestão: br_inep_avaliacao_alfabetizacao_municipio.csv
Tabela salva: workspace.bronze.br_inep_avaliacao_alfabetizacao_municipio
Total registros: 23995
Ingestão finalizada: br_inep_

# Visualizando os Dados

In [0]:
# ============================================================
# VALIDAÇÃO FINAL
# ============================================================

tabelas_bronze = [
    "br_inep_avaliacao_alfabetizacao_municipio",
    "br_inep_avaliacao_alfabetizacao_uf",
    "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil",
    "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf",
    "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio"
]

for tabela in tabelas_bronze:

    print("=" * 80)
    print(f"Tabela: {tabela}")

    df = spark.table(f"workspace.bronze.{tabela}")

    print(f"Total registros: {df.count()}")

    display(df.limit(5))

Tabela: br_inep_avaliacao_alfabetizacao_municipio
Total registros: 23995


ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada
2023,1100031,2,3,69.1,767.8763,null,null,null,null,null,null,null,null,null,0,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,bronze
2023,1100072,2,3,58.2,747.8918,null,null,null,null,null,null,null,null,null,1,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,bronze
2023,1100189,2,5,69.73,762.4062,null,null,null,null,null,null,null,null,null,2,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,bronze
2023,1101609,2,3,50.7,745.6802,null,null,null,null,null,null,null,null,null,3,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,bronze
2023,1101807,2,3,55.69,752.3724,null,null,null,null,null,null,null,null,null,4,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,bronze


Tabela: br_inep_avaliacao_alfabetizacao_uf
Total registros: 145


ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada
2023,AM,2,3,49.2,733.6637,null,null,null,null,null,null,null,null,null,0,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,bronze
2023,PB,2,2,55.23,744.8152,null,null,null,null,null,null,null,null,null,1,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,bronze
2023,PR,2,5,73.12,757.2146,null,null,null,null,null,null,null,null,null,2,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,bronze
2023,AP,2,3,41.87,732.7858,null,null,null,null,null,null,null,null,null,3,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,bronze
2023,PE,2,5,58.95,747.4522,null,null,null,null,null,null,null,null,null,4,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,bronze


Tabela: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil
Total registros: 3


ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada
2025,Pública,66,60,64,67,71,74,77,80,88,0,2026-06-24T13:45:15.840Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv,bronze
2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80,87.37,1,2026-06-24T13:45:15.840Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv,bronze
2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80,86,2,2026-06-24T13:45:15.840Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv,bronze


Tabela: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf
Total registros: 54


ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada
2024,RR,Pública,null,null,null,null,null,null,null,null,null,0,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,bronze
2023,RR,Pública,null,null,null,null,null,null,null,null,null,1,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,bronze
2023,AC,Pública,null,null,56.9,62.2,67.3,72,76.2,80,null,2,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,bronze
2024,AC,Pública,51.38,null,56.9,62.2,67.3,72,76.2,80,80.87,3,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,bronze
2023,DF,Pública,null,null,63.1,67,70.6,74,77.1,80,null,4,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,bronze


Tabela: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio
Total registros: 10704


ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada
2023,4301750,Municipal,null,null,14.05,23.65,37,52.68,67.85,80,null,null,0,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,bronze
2024,4301750,Municipal,4.4,null,14.05,23.65,37,52.68,67.85,80,0,92.59,1,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,bronze
2024,2406908,Municipal,42.86,7.94,14.05,23.65,37,52.68,67.85,80,1,84,2,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,bronze
2023,2406908,Municipal,4.4,7.94,14.05,23.65,37,52.68,67.85,80,0,82.14,3,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,bronze
2023,1718501,Municipal,4.6,8.25,14.48,24.16,37.49,53.03,68,80,0,95.65,4,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,bronze
